# Generate `mimic_cxr_cleaned.csv` (nguồn: Google Cloud Storage)

Chạy notebook này **một lần** để tạo `mimic_cxr_cleaned.csv` — bảng ghép `dicom_id → findings → đường dẫn ảnh` cho **toàn bộ p10–p19**.

**Nguồn dữ liệu: chỉ GCS** — `gs://mimic-cxr-jpg-dataset-kaggle`. Không cần đính kèm Kaggle dataset ảnh/report nữa.

Notebook sẽ:
1. Xác thực GCS — Kaggle: dùng secret `GCS_SERVICE_ACCOUNT`; GCP VM / local: dùng `gcloud auth` sẵn có.
2. `gsutil rsync` toàn bộ báo cáo `.txt` + tải `metadata.csv.gz` từ bucket.
3. Parse FINDINGS (fallback IMPRESSION), merge với metadata, sinh đường dẫn ảnh `pXX/pSUBJECT/sSTUDY/dicom.jpg`.
4. Lưu `mimic_cxr_cleaned.csv` — 5 cột: `dicom_id, findings, Img_Folder, Img_Filename, Note_file`.

**Yêu cầu:** `gsutil`/`gcloud` (có sẵn trên Kaggle & GCP VM). Trên Kaggle: thêm secret `GCS_SERVICE_ACCOUNT` (Settings → Add-ons → Secrets) và bật Internet.

In [ ]:
# ============================================================================
# Notebook 01 — Generate mimic_cxr_cleaned.csv  (GCS-only)
# Đọc reports + metadata trực tiếp từ gs://mimic-cxr-jpg-dataset-kaggle.
# Không cần Kaggle image/report dataset.
# ----------------------------------------------------------------------------
# Cell 1: cấu hình, xác thực GCS, tải reports + metadata về máy chạy.
# ============================================================================
import os
import subprocess
import tempfile

# ---- Config (đổi qua biến môi trường nếu cần) ----
GCS_BUCKET   = os.environ.get("GCS_DATA_BUCKET", "gs://mimic-cxr-jpg-dataset-kaggle")
REPORTS_GCS  = f"{GCS_BUCKET}/mimic-cxr-reports/files"
METADATA_GCS = f"{GCS_BUCKET}/mimic-cxr-2.0.0-metadata.csv.gz"


def _default_workdir():
    return "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./mimic_prep")


WORK_DIR = os.environ.get("WORK_DIR", _default_workdir())
os.makedirs(WORK_DIR, exist_ok=True)

# Đường ra: ưu tiên biến môi trường, rồi configs/kaggle_datasets.yaml, cuối cùng mặc định.
OUTPUT_CSV = os.environ.get("CLEANED_CSV_PATH")
if not OUTPUT_CSV:
    try:
        import yaml
        with open("configs/kaggle_datasets.yaml") as f:
            OUTPUT_CSV = yaml.safe_load(f)["working"]["cleaned_csv_path"]
    except Exception:
        OUTPUT_CSV = os.path.join(WORK_DIR, "mimic_cxr_cleaned.csv")

LOCAL_REPORTS = os.path.join(WORK_DIR, "reports_txt")
LOCAL_META    = os.path.join(WORK_DIR, "mimic-cxr-2.0.0-metadata.csv.gz")

print(f"GCS bucket : {GCS_BUCKET}")
print(f"Output CSV : {OUTPUT_CSV}")
print(f"Work dir   : {WORK_DIR}")


# ---- Xác thực GCS ----
# Kaggle: secret GCS_SERVICE_ACCOUNT (chuỗi JSON). VM/local: dùng gcloud auth sẵn có.
def ensure_gcs_auth():
    sa_json = os.environ.get("GCS_SERVICE_ACCOUNT")
    if not sa_json:
        try:
            from kaggle_secrets import UserSecretsClient
            sa_json = UserSecretsClient().get_secret("GCS_SERVICE_ACCOUNT")
        except Exception:
            sa_json = None
    if sa_json:
        key_path = os.path.join(tempfile.gettempdir(), "gcs_sa.json")
        with open(key_path, "w") as f:
            f.write(sa_json)
        subprocess.run(
            ["gcloud", "auth", "activate-service-account", "--key-file", key_path],
            check=True,
        )
        os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = key_path
        print("Đã xác thực bằng GCS service account.")
    else:
        print("Không có GCS_SERVICE_ACCOUNT — dùng gcloud credentials sẵn có.")


ensure_gcs_auth()

# ---- Tải reports (.txt) + metadata từ GCS ----
# ~227k file text nhỏ → rsync song song (-m). Chạy một lần.
os.makedirs(LOCAL_REPORTS, exist_ok=True)
print("Đang sync báo cáo từ GCS (có thể mất vài phút)…")
subprocess.run(["gsutil", "-m", "-q", "rsync", "-r", REPORTS_GCS, LOCAL_REPORTS], check=True)
print("Đang tải metadata CSV…")
subprocess.run(["gsutil", "-q", "cp", METADATA_GCS, LOCAL_META], check=True)
print("Tải xong.")


In [ ]:
# ============================================================================
# Cell 2: parse reports → merge metadata → lưu mimic_cxr_cleaned.csv
# ============================================================================
import os
import re
import glob
import pandas as pd
from multiprocessing import Pool, cpu_count

# ---- Bóc tách section báo cáo (ưu tiên FINDINGS, fallback IMPRESSION) ----
SECTION_PAT = re.compile(
    r"(FINAL REPORT|EXAMINATION|INDICATION|TECHNIQUE|COMPARISON|HISTORY|"
    r"FINDINGS|IMPRESSION|RECOMMENDATION|NOTIFICATION|CLINICAL HISTORY|REASON FOR EXAMINATION|"
    r"WET READ|WET READ VERSION):",
    re.IGNORECASE,
)


def extract_findings(text):
    parts = SECTION_PAT.split(text)
    sections = {}
    for i in range(1, len(parts), 2):
        sections[parts[i].upper().strip()] = parts[i + 1].strip()
    return sections.get("FINDINGS") or sections.get("IMPRESSION") or ""


def parse_one_report(txt_path):
    try:
        study_id = int(os.path.basename(txt_path)[1:-4])  # 'sXXXXXXXX.txt' -> int
        with open(txt_path, encoding="utf-8") as f:
            findings = extract_findings(f.read()).replace("\n", " ").strip()
    except (ValueError, OSError):
        return None
    if not findings:
        return None
    return (study_id, findings, os.path.basename(txt_path))


# ---- Parse toàn bộ báo cáo song song ----
txt_files = glob.glob(f"{LOCAL_REPORTS}/**/s*.txt", recursive=True)
print(f"Tìm thấy {len(txt_files)} file báo cáo .txt")
if not txt_files:
    raise FileNotFoundError(f"Không có báo cáo trong {LOCAL_REPORTS}. rsync GCS đã chạy chưa?")

n_workers = max(1, cpu_count())
print(f"Parse với {n_workers} CPU workers…")
with Pool(processes=n_workers) as pool:
    results = pool.map(parse_one_report, txt_files, chunksize=500)

reports = pd.DataFrame(
    [r for r in results if r is not None],
    columns=["study_id", "findings", "Note_file"],
)
print(f"Bóc tách được {len(reports)} báo cáo có FINDINGS")

# ---- Merge với metadata (bao trùm p10–p19) ----
metadata = pd.read_csv(LOCAL_META)  # pandas tự giải nén .gz
df = metadata.merge(reports, on="study_id", how="inner")

# ---- Sinh đường dẫn ảnh: pXX/pSUBJECT/sSTUDY/dicom.jpg ----
subj = df["subject_id"].astype(str)
sid = df["study_id"].astype(str)
df["Img_Folder"] = "p" + subj.str[:2] + "/p" + subj + "/s" + sid
df["Img_Filename"] = df["dicom_id"].astype(str) + ".jpg"

# Bucket GCS là full dataset nên mọi dòng metadata đều có ảnh tương ứng
# → không cần bước quét ảnh tồn tại trên đĩa như luồng Kaggle-subset cũ.
df = df[["dicom_id", "findings", "Img_Folder", "Img_Filename", "Note_file"]]

os.makedirs(os.path.dirname(OUTPUT_CSV) or ".", exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nĐã lưu {OUTPUT_CSV}: {len(df):,} dòng (toàn bộ nhóm bệnh nhân)")
print(df[["Img_Folder", "Img_Filename"]].head(3).to_string())
